# ArcGIS Workforce → Flat Feature Service

This notebook takes the default ArcGIS Workforce feature service and publishes a single hosted Feature Layer containing fields from the Assignments, Workers, Dispatchers, and Assignment Types layers in the ArcGIS Workforce feature service.

Note: If you have made changes to the feature service layers you may need to modify this notebook to account for them.

## Workforce schema (default)

| Service index | Type | Layer name | Primary key |
|---|---|---|---|
| 0 | Feature Layer | Workers | `GlobalID` |
| 1 | Feature Layer | Assignments | `GlobalID` (FKs: `workerid`, `dispatcherid`, `assignmenttype`) |
| 2 | Table | Dispatchers | `GlobalID` |
| 3 | Table | Assignment Types | `GlobalID` |
| 4 | Table | Assignment Integrations | `GlobalID` (FK: `assignmenttype`) |

**Join strategy:** Assignments is the spine. All other tables are left-joined onto it using `GlobalID`-based foreign keys so that every assignment row is preserved regardless of whether related records exist. The Integrations table is not included in the final flat feature service.

---

## 1 — Import libraries

In [ ]:
#from osgeo import gdal
#import shapefile
import pandas as pd
import arcgis, sys
from arcgis.gis import GIS
from arcgis.features import FeatureLayerCollection#, FeatureSet
#from arcgis.features import GeoAccessor, GeoSeriesAccessor  # required for .spatial accessor

print(f'Python  {sys.version}')
print(f'arcgis  {arcgis.__version__}')

## 2 — Configuration
Enter username when running cell below, you will be prompted for the password when running cell 3

In [ ]:
# ── Connection ───────────────────────────────────────────────────────────────
portal_url = input('Enter ArcGIS Online or Enterprise URL: ') # (e.g., https://www.arcgis.com, https://YOURORG.maps.arcgis.com)
username   = input('Enter username: ')

# ── Workforce item ───────────────────────────────────────────────────────────
# The 32-character hex item ID from the Workforce feature service item page URL.
# Open the item page in AGOL: …/home/item.html?id=<THIS PART>
workforce_item_id = input('Enter Workforce feature service ID: ') # (e.g., 4rts7307f134432ah384rt5e8dnh838c)

# ── Output ───────────────────────────────────────────────────────────────────
output_title  = input('Enter output feature service title: ') # Add new FS title (e.g., 'Workforce Joined')
output_name = input('Enter output layer name: ') # Add a title for the layer (e.g. 'Joined Assignments')
OUTPUT_TAGS   = 'workforce, flat, joined'
OUTPUT_FOLDER = None   # None = root content folder; or set to a folder name string

## 3 — Connect to portal

In [ ]:
# Leave USERNAME / PASSWORD empty to use an interactive login prompt instead.
gis = GIS(portal_url, username)
#gis = GIS('home') #use this when running as a hosted notebook in AGOL/Enterprise
print(f'Connected as: {gis.properties.user.username}')

## 4 — Load the Workforce feature service

In [ ]:
wf_item = gis.content.get(workforce_item_id)
assert wf_item is not None, 'Item not found. Verify workforce_item_id and your permissions.'

print(f'Title : {wf_item.title}')
print(f'Type  : {wf_item.type}')
print(f'URL   : {wf_item.url}')

flc = FeatureLayerCollection.fromitem(wf_item)

print('\nFeature Layers:')
for i, lyr in enumerate(flc.layers):
    print(f'  [{i}] {lyr.properties.name}')

print('\nTables:')
for i, tbl in enumerate(flc.tables):
    print(f'  [{i}] {tbl.properties.name}')

## 5 — Resolve layer and table references

Default, unmodified Workforce service layer order:
- `layers[0]` → **Assignments**
- `layers[1]` → **Workers**
- `tables[0]` → **Dispatchers**
- `tables[1]` → **Assignment Types**
- `tables[2]` → **Assignment Integrations**

Verify that the layers and tables have the correct as reported above. If there is a mismatch, use the next cell to remap them to the appropriate variable.

In [ ]:
assignments_lyr  = flc.layers[0]   # Assignments
workers_lyr      = flc.layers[1]   # Workers
dispatchers_tbl  = flc.tables[0]   # Dispatchers
assign_types_tbl = flc.tables[1]   # Assignment Types
integrations_tbl = flc.tables[2]   # Assignment Integrations

# Feature layers
print(f'Assignments          : {assignments_lyr.properties.name}')
print(f'Workers              : {workers_lyr.properties.name}')
print(f'Dispatchers          : {dispatchers_tbl.properties.name}')

# Tables
print(f'Assignment Types     : {assign_types_tbl.properties.name}')
print(f'Assignment Integrations: {integrations_tbl.properties.name}')

## 6 — Collect domain definitions from all source layers

Each source layer's field list is inspected for non-null `domain` properties. The domain definition is stored keyed by the **prefixed** field name that will exist in the flat output (e.g. the `status` field in the Assignments layer becomes `asgn_status` in the flat layer).

In [ ]:
def collect_domains(layer, prefix):
    '''
    Inspect all fields in `layer` and return a dict mapping
    prefixed_field_name → domain_dict for every field that has a domain.

    The domain_dict has the structure the REST updateDefinition endpoint expects:
        {
            'type': 'codedValue',
            'name': '<domain_name>',
            'codedValues': [{'name': '<label>', 'code': <value>}, ...]
        }
    '''
    domains = {}
    for field in layer.properties.fields:
        domain = field.get('domain')
        if domain and domain.get('type') == 'codedValue':
            prefixed_name = f'{prefix}_{field['name']}'
            domains[prefixed_name] = {
                'type': 'codedValue',
                'name': domain['name'],
                'codedValues': [
                    {'name': cv['name'], 'code': cv['code']}
                    for cv in domain['codedValues']
                ],
            }
    return domains


# Collect domains from every source layer using the same prefix scheme
# that will be applied to the DataFrames later.
all_domains = {}
all_domains.update(collect_domains(workers_lyr,      'wrkr'))
all_domains.update(collect_domains(assignments_lyr,  'asgn'))
all_domains.update(collect_domains(dispatchers_tbl,  'disp'))
all_domains.update(collect_domains(assign_types_tbl, 'type'))
all_domains.update(collect_domains(integrations_tbl, 'intg'))

print(f'Found {len(all_domains)} field(s) with coded value domains:\n')
for field_name, domain in all_domains.items():
    values_preview = ', '.join(
        f"{cv['code']}='{cv['name']}'" for cv in domain['codedValues']
    )
    print(f'  {field_name}')
    print(f'    Domain : {domain['name']}')
    print(f'    Values : {values_preview}')
    print()

## Cell 6 — Query function

This function sets the parameters for the feature service query.

In [ ]:
def query_all(layer, with_geometry=False):
    
    name = layer.properties.name

    fs = layer.query(
        where='1=1',
        out_fields='*',
        return_geometry=with_geometry,
        out_sr=4326 if with_geometry else None,
        return_all_records=True
    )
    result = fs.sdf
    print(f'  {name}: {len(result)} records.          ')
    return result

## 8 — Fetch all data from the service

In [ ]:
print('Fetching Assignments (with geometry)…')
df_assignments = query_all(assignments_lyr, with_geometry=True)

print('Fetching Workers…')
df_workers = query_all(workers_lyr)

print('Fetching Dispatchers…')
df_dispatchers = query_all(dispatchers_tbl)

print('Fetching Assignment Types…')
df_assign_types = query_all(assign_types_tbl)

print('\nRecord counts:')
print(f'  Assignments            : {len(df_assignments)}')
print(f'  Workers                : {len(df_workers)}')
print(f'  Dispatchers            : {len(df_dispatchers)}')
print(f'  Assignment Types       : {len(df_assign_types)}')

## 9 — Prefix columns to prevent name collisions

Every column is renamed `<prefix>_<OriginalFieldName>` before merging. This prevents collisions between identically named fields that appear across multiple tables (e.g. `GlobalID`, `status`, `CreationDate`).

In [ ]:
def prefix_columns(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    '''Rename every column to prefix_OriginalName.'''
    return df.rename(columns={c: f'{prefix}_{c}' for c in df.columns if c != 'SHAPE'})

# ── Prefix ───────────────────────────────────────────────────────────────────
#df_asgn = prefix_columns(df_assignments_geo, 'asgn')  # Assignments
df_asgn = prefix_columns(df_assignments,     'asgn')  # Assignments
df_wrkr = prefix_columns(df_workers,         'wrkr')  # Workers
df_disp = prefix_columns(df_dispatchers,     'disp')  # Dispatchers
df_type = prefix_columns(df_assign_types,    'type')  # Assignment Types

print('\nColumn counts after prefixing:')
for label, df in [('asgn (Assignments)', df_asgn), ('wrkr (Workers)', df_wrkr),
                  ('disp (Dispatchers)', df_disp), ('type (Assign. Types)', df_type)]:
    print(f'  {label}: {len(df.columns)} columns')

In [ ]:
# ── Diagnostic: print all prefixed column names before joining ───────────────
print('=== df_asgn columns (Assignments) ===')
for col in sorted(df_asgn.columns):
    print(f'  {col}')

print('\n=== df_wrkr columns (Workers) ===')
for col in sorted(df_wrkr.columns):
    print(f'  {col}')

print('\n=== df_disp columns (Dispatchers) ===')
for col in sorted(df_disp.columns):
    print(f'  {col}')

print('\n=== df_type columns (Assignment Types) ===')
for col in sorted(df_type.columns):
    print(f'  {col}')

## 10 — Join all tables onto Assignments

Joins follow the relationships defined in the official Workforce schema documentation:

| Join | Left key (Assignments) | Right key |
|---|---|---|
| → Workers | `asgn_workerid` | `wrkr_GlobalID` |
| → Dispatchers | `asgn_dispatcherid` | `disp_GlobalID` |
| → Assignment Types | `asgn_assignmenttype` | `type_GlobalID` |

All joins are `left` so every Assignment row is retained.

In [ ]:
flat = (
    df_asgn

    # Assignments.workerid → Workers.GlobalID
    .merge(
        df_wrkr,
        left_on='asgn_workerid',
        right_on='wrkr_GlobalID',
        how='left',
    )

    # Assignments.dispatcherid → Dispatchers.GlobalID
    .merge(
        df_disp,
        left_on='asgn_dispatcherid',
        right_on='disp_GlobalID',
        how='left',
    )

    # Assignments.assignmenttype → AssignmentTypes.GlobalID
    .merge(
        df_type,
        left_on='asgn_assignmenttype',
        right_on='type_GlobalID',
        how='left',
    )
)

print(f'Joined: {flat.shape[0]} rows × {flat.shape[1]} columns')
print(f'Expected rows: {len(df_assignments)} (one per assignment)')

## 11 — Clean up redundant columns

In [ ]:
# Drop the right-side join key columns — they duplicate values already in asgn_
redundant = ['wrkr_GlobalID', 'disp_GlobalID', 'type_GlobalID', 'intg_assignmenttype']
flat.drop(columns=redundant, inplace=True, errors='ignore')

print(f'Final flat DataFrame: {flat.shape[0]} rows × {flat.shape[1]} columns')
flat.head(3)

## 12 — Verify domain fields exist in the flat DataFrame and review all output columns

Cross-checks that every domain-carrying field collected in Cell 6 actually landed in the flat output. Any mismatch is reported here before publish so you can investigate.

In [ ]:
flat_cols = set(flat.columns)
domains_to_apply   = {}   # prefixed_field_name → domain_dict  (fields confirmed present)
domains_not_found  = {}   # prefixed_field_name → domain_dict  (fields not in flat output)

for prefixed_field, domain in all_domains.items():
    if prefixed_field in flat_cols:
        domains_to_apply[prefixed_field] = domain
    else:
        domains_not_found[prefixed_field] = domain

print(f'Domains to apply   : {len(domains_to_apply)}')
for f, d in domains_to_apply.items():
    print(f'  ✅  {f}  ({d['name']})')

if domains_not_found:
    print(f'\nDomains NOT found in flat output (will be skipped): {len(domains_not_found)}')
    for f, d in domains_not_found.items():
        print(f'  ⚠️  {f}  ({d['name']})')
else:
    print('\n✅  All domain fields present in flat output.')

print(f'Total columns: {len(flat.columns)}\n')
for col in flat.columns:
    print(f'  {col}')

## 13 — (Optional) Save a local CSV snapshot

In [ ]:
csv_path = 'Backup.csv' # Enter desired file name
flat.to_csv(csv_path, index=False)
print(f'CSV saved → {csv_path}  ({len(flat)} rows, {len(flat.columns)-1} columns)')

## 14 — Publish as a new hosted Feature Layer

In [ ]:
published = flat.spatial.to_featurelayer(
    title=output_title,
    gis=gis,
    tags=OUTPUT_TAGS,
    folder=OUTPUT_FOLDER,
    service_name=output_name
)

print(f'\n✅  Published successfully!')
print(f'   Item ID  : {published.id}')
print(f'   Item URL : {portal_url}/home/item.html?id={published.id}')
published

## 15 — Apply coded value domains to the published layer

The `spatial.to_featurelayer()` call publishes raw data with no domain definitions. This cell calls `layer.manager.update_definition()` with the full domain payload for every field that had a domain in the source service.

The update dict structure required by the REST API is:
```json
{
  'fields': [
    {
      'name': '<prefixed_field_name>',
      'domain': {
        'type': 'codedValue',
        'name': '<domain_name>',
        'codedValues': [{'name': '<label>', 'code': <value>}, ...]
      }
    }
  ]
}
```

In [ ]:
# Get the published feature layer object
published_flc = FeatureLayerCollection.fromitem(published)
flat_layer    = published_flc.layers[0] # if published_flc.layers else published_flc.tables[0]

# Build the update_definition payload — one entry per domain field
fields_payload = [
    {
        'name':   prefixed_field,
        'domain': domain_dict,
    }
    for prefixed_field, domain_dict in domains_to_apply.items()
]

if not fields_payload:
    print('No domains to apply — skipping update_definition call.')
else:
    print(f'Applying {len(fields_payload)} domain(s) to the published layer…')
    result = flat_layer.manager.update_definition({'fields': fields_payload})
    print(f'update_definition response: {result}')

    # Confirm by re-reading the layer definition and checking domain fields
    confirmed, missing = [], []
    field_map = {f['name']: f for f in flat_layer.properties.fields}

    for prefixed_field in domains_to_apply:
        field_def = field_map.get(prefixed_field, {})
        if field_def.get('domain'):
            confirmed.append(prefixed_field)
        else:
            missing.append(prefixed_field)

    print(f'\nDomain confirmation ({len(confirmed)} applied, {len(missing)} missing):')
    for f in confirmed:
        print(f'  ✅  {f}')
    for f in missing:
        print(f'  ⚠️  {f}  — domain not confirmed on published layer')

## 16 — Attachments

If your assignments layer contains attachments, they will need to be downlaoded locally and then re-uploaded to the final layer.

In [ ]:
# Enable attachments on the new layer
flat_layer.manager.update_definition({'hasAttachments':'true'})

def query_attachments(assignmentslayer):
    
    attach = assignmentslayer.attachments
    all_attach = attach.search(where='1=1')
    return all_attach

def move_attachments(assignmentslayer, flatlayer, lyr_attachments):
    feature_count = len(lyr_attachments)
    current = 0
    for attachment in lyr_attachments:
        parent_oid = attachment['PARENTOBJECTID']
        att_id = attachment['ID']

        downloaded_path = assignmentslayer.attachments.download(oid=parent_oid, attachment_id=att_id, save_path='./temp_attachments')

        #Find corresponding feature in target layer 
        target_feature = flatlayer.query(where=f'asgn_OBJECTID = {parent_oid}').features[0]
        target_oid = target_feature.attributes['OBJECTID']
    
        #upload to target layer
        current += 1
        print(f'Uploading attachment {current} of {feature_count}')
        flat_layer.attachments.add(oid=target_oid, file_path=downloaded_path[0])


attachments = query_attachments(assignments_lyr)

if attachments:
    print('Uploading attachments to new service')
    move_attachments(assignments_lyr, flat_layer, attachments)
    print('Attachments uploaded')
else: print('No attachments found')

---
## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `Item not found` | Wrong item ID or insufficient sharing | Verify `WORKFORCE_ITEM_ID` and that you have access to the item |
| Layer name in Cell 5 doesn't match expected | Service is a Classic Workforce project | Classic projects use a different schema; see Esri docs for Classic field names |
| `spatial.to_featurelayer` error | SHAPE column contains null geometries | Add `flat = flat.dropna(subset=['SHAPE'])` before Cell 13 |
| ArcGIS Enterprise field name differences | Enterprise uses lowercase system fields (`objectid`, `globalid`, `created_date`, etc.) | The `.sdf` reflects this automatically; the join keys `workerid`, `dispatcherid`, `assignmenttype` are unaffected |
|'This service name is unavailable for Feature Service'|The output title or name entered in Cell 2 already exist | Re-run cell 2 and enter unique names for the title and name.|